#Part 1 (10 points):
Implement the scaled dot-product attention as discussed in class
(lecture 10) from scratch (use NumPy and pandas only, no deep learning libraries are
allowed for this step).

In [ ]:
import numpy as np

def softmax(x, axis=-1):
    e_x = np.exp(x-np.max(x, axis=axis, keepdims=True))
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    # find d_k
    d_k = Q.shape[-1]

    # calculate QK^T
    QKT = np.matmul(Q, np.transpose(K, -1, -2)) / np.sqrt(d_k)

    # apply mask
    if mask is not None:
        QKT = np.where(mask == 0, -np.inf, QKT)

    # normalize
    QKT = softmax(QKT)

    # compute attention matrix
    A = np.matmul(QKT, V)

    return A, QKT

In [ ]:
# torch equivalent to be integrated into later parts
import torch
import torch.nn as nn
import torch.nn.functional as F

def scaled_dot_product_attention_torch(Q, K, V, mask=None):
    d_k = Q.size(-1)

    QKT = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)

    # apply mask
    if mask is not None:
        QKT = QKT.masked_fill(mask == 0, -np.inf)

    # normalize
    QKT = F.softmax(QKT, dim=-1)

    # compute attention matrix
    A = torch.matmul(QKT, V)

    return A, QKT

#Part 2 (10 points):
Pick any encoder-decoder seq2seq model (as discussed in class) and
integrate the scaled dot-product attention in the encoder architecture. You may come
up with your own technique of integration or adopt one from literature. Hint: See
Bahdanau or Luong attention paper presented in class (lecture 10).


In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()
        # tranlate word tokens ids into dense math vectors
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hidden_dim, batch_first=True)

    def forward(self, src):
        embedded = self.embedding(src)
        # output has the hidden state or every word
        # hidden has the final hidden shape
        outputs, hidden = self.rnn(embedded)

        # outputs is to act as keys and values for decoders attention
        return outputs, hidden

In [ ]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        # gru input will be current word embedding + attention context vector
        self.rnn = nn.GRU(emb_dim + hidden_dim, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, input, hidden, encoder_outputs):
        # only process 1 word at a time
        embedded = self.embedding(input)

        # the decoder's previous hidden state
        Q = hidden.permute(1, 0, 2)
        # encoder's output for source sentence
        K = encoder_outputs
        V = encoder_outputs

        # use the torch scaled dot product attention I made
        context, attention_weights = scaled_dot_product_attention_torch(Q, K, V)

        # combine embedded input word with context
        rnn_input = torch.cat((embedded, context), dim=2)

        # generate next hidden state
        output, hidden = self.rnn(rnn_input, hidden)

        # predict actual word id from vocab
        prediction = self.fc_out(output.squeeze(1))

        return prediction, hidden

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.fc_out.out_features

        # empty tensor to save decoder predictions
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)

        # pass whole source through encoder
        encoder_outputs, hidden = self.encoder(src)

        # get the first token from the target (<SOS>)
        input = trg[:, 0].unsqueeze(1)

        # generate words one at a time
        for t in range(1, trg_len):
            # pass everything into the decoder
            output, hidden = self.decoder(input, hidden, encoder_outputs)
            # save the prediction
            outputs[:, t, :] = output

            # feed the actually correct word occasionally to ensure we're on track
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            # get id of word with best prob
            top1 = output.argmax(1)
            input = trg[:, t].unsqueeze(1) if teacher_force else top1.unsqueeze(1)

        return outputs

#Part 3 (5 points):
Pick any public dataset of your choice (use a small-scale dataset like a
subset of the Tatoeba or Multi30k dataset) for machine translation task. Train your
model from Part 2 for the machine translation task. Evaluate test set by reporting the
BLEU Score.


In [ ]:
!pip install datasets nltk

In [ ]:
from datasets import load_dataset

import nltk
from nltk.translate.bleu_score import corpus_bleu

import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence

from collections import Counter

import random

In [ ]:
# load the tatoeba dataset from huggingface, for this assignment we will get 10,000 sentences
dataset = load_dataset("opus_books", "en-fr", split="train[:10000]")

# basic tokenization by forcing to lowercase and splitting by spaces
eng_sentences = [item['translation']['en'].lower().split() for item in dataset]
fra_sentences = [item['translation']['fr'].lower().split() for item in dataset]

README.md: 0.00B [00:00, ?B/s]

en-fr/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/127085 [00:00<?, ? examples/s]

In [ ]:
# build a basic vocab
class Vocabulary:
    def __init__(self, freq_threshold=2):
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.stoi = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        # minimum number of times to see word to add to vocab
        self.freq_threshold = freq_threshold

    def __len__(self):
        return len(self.itos)

    def build_vocabulary(self, sentence_list):
        frequencies = Counter()
        idx = 4 # there are already 3 reserved
        for sentence in sentence_list:
            for word in sentence:
                frequencies[word] += 1
                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1

    def numericalize(self, text):
        # convert list of words into list of integers with UNK if not in vocab
        return [self.stoi.get(token, self.stoi["<UNK>"]) for token in text]

eng_vocab = Vocabulary()
fra_vocab = Vocabulary()

eng_vocab.build_vocabulary(eng_sentences)
fra_vocab.build_vocabulary(fra_sentences)

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, src_sentences, trg_sentences, src_vocab, trg_vocab):
        self.src_sentences = src_sentences
        self.trg_sentences = trg_sentences
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab

    def __len__(self):
        return len(self.src_sentences)

    def __getitem__(self, index):
        # when fettching, wrap in SOS and EOS for start and stop
        src = [self.src_vocab.stoi["<SOS>"]] + self.src_vocab.numericalize(self.src_sentences[index]) + [self.src_vocab.stoi["<EOS>"]]
        trg = [self.trg_vocab.stoi["<SOS>"]] + self.trg_vocab.numericalize(self.trg_sentences[index]) + [self.trg_vocab.stoi["<EOS>"]]
        return torch.tensor(src), torch.tensor(trg)

In [ ]:
# ensure every sentences is exactly as long as the longest sentence
def collate_fn(batch):
    src_batch, trg_batch = [], []
    for src_item, trg_item in batch:
        src_batch.append(src_item)
        trg_batch.append(trg_item)

    src_batch = pad_sequence(src_batch, padding_value=0, batch_first=True)
    trg_batch = pad_sequence(trg_batch, padding_value=0, batch_first=True)

    return src_batch, trg_batch

In [ ]:
# complete the data pre-processing
dataset = TranslationDataset(eng_sentences, fra_sentences, eng_vocab, fra_vocab)
train_size = int(0.9 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

In [ ]:
# initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

INPUT_DIM = len(eng_vocab)
OUTPUT_DIM = len(fra_vocab)
EMB_DIM=64
HID_DIM=128

encoder = Encoder(INPUT_DIM, EMB_DIM, HID_DIM).to(device)
decoder = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM).to(device)
model = Seq2Seq(encoder, decoder, device).to(device)

optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=0)

In [ ]:
# model training
epochs = 10

for epoch in range(epochs):
    model.train()
    epoch_loss = 0

    for src, trg in train_loader:
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()

        output = model(src, trg)
        output_dim = output.shape[-1]

        output = output[:, 1:].reshape(-1, output_dim)
        trg = trg[:, 1:].reshape(-1)

        loss = criterion(output, trg)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch: {epoch+1}, Loss: {epoch_loss/len(train_loader):.3f}")

Epoch: 1, Loss: 6.767
Epoch: 2, Loss: 6.443
Epoch: 3, Loss: 6.259
Epoch: 4, Loss: 6.095
Epoch: 5, Loss: 5.942
Epoch: 6, Loss: 5.811
Epoch: 7, Loss: 5.709
Epoch: 8, Loss: 5.602
Epoch: 9, Loss: 5.520
Epoch: 10, Loss: 5.432


In [ ]:
# BLEU Eval
def evaluate_bleu(model, test_loader, trg_vocab):
    model.eval()
    targets = []
    outputs = []

    with torch.no_grad():
        for src, trg in test_loader:
            src = src.to(device)

            # no teacher forcing for eval
            output = model(src, trg = trg.to(device), teacher_forcing_ratio=0.0)

            # get the predicted tokens
            pred_tokens = output.argmax(2)

            for i in range(trg.shape[0]):
                # recontruct the true target sentence
                trg_words = [trg_vocab.itos[idx.item()] for idx in trg[i] if idx.item() not in [0, 1, 2]]
                targets.append([trg_words])

                # reconstruct the model's predicted sentence
                pred_words = []
                for idx in pred_tokens[i][1:]:
                    if idx.item() == 2:
                        break
                    if idx.item() not in [0, 1]:
                        pred_words.append(trg_vocab.itos.get(idx.item(), "<UNK>"))
                outputs.append(pred_words)

    bleu_score = corpus_bleu(targets, outputs)
    return bleu_score

In [ ]:
bleu = evaluate_bleu(model, test_loader, fra_vocab)
print(f"BLEU Score: {bleu*100:.2f}")

BLEU Score: 0.44


#Part 4 (30 points):
In this part you are required to implement a simplified Transformer
model from scratch (using Python and NumPy/PyTorch/TensorFlow with minimal highlevel abstractions) and apply it to a machine translation task (e.g., English-to-French or
English-to-German translation) using the same dataset from part 3.

We discussed Transformer architecture in depth in class (Vaswani Paper – Attention is
all you need). Apply the following simplifications to the original model architecture:
1. Reduced Model Depth: Use 2 encoder layers and 2 decoder layers instead of
the standard 6.
2. Limited Attention Heads: Use 2 attention heads in the multi-head attention
mechanism rather than 8.
3. Smaller Embedding Size: Set the embedding dimension to 64 instead of 512.
4. Reduced Feedforward Network Size: Use a feedforward dimension of 128
instead of 2048.
5. Smaller Dataset: Use a small dataset (e.g., about 10k sentence pairs).
6. Tokenization Simplifications: Use a basic subword tokenizer (like Byte Pair
Encoding - BPE) or word-level tokenization instead of complex language-specific
tokenizers.

Key components to implement:
1. Positional Encoding: Implement Sinusoidal position encoding.
2. Scaled dot-product attention: Use the same implementation from part 1.
Projects in Machine Learning and AI (RPI Spring 2026)
3. Multi-Head Attention: Integrate the scaled dot-product attention into a multihead attention framework using the specified simplifications.
4. Encoder and Decoder Blocks: Implement simplified encoder and decoder
layers, ensuring: Layer normalization, Residual connections, Masked attention in
the decoder for autoregressive generation.
5. Final Output Layer: Implement a linear layer followed by a SoftMax activation
for generating translated tokens.

Evaluation: Compute the BLEU score on a validation set and compare the performance
with your model from part 2. Explain why there are differences in performance. Also
discuss any other differences you notice, for example runtime etc.

In [ ]:
import math
# add unique patterns to the embeddings so an understanding of order is maintained
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        # zero matrix
        pe = torch.zeros(max_len, d_model)
        # column vector of all positions
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # divisor term
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # save the state, but not as a learnable param
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        # add pe to the input embeddings matching the length of x
        return x + self.pe[:, :x.size(1), :]

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=64, num_heads=2):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        # get the dimension of each head
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        # mix the heads back together
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)

        # pass through the linear layers, reshape, and transpose
        Q = self.W_q(q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(k).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(v).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)


        output, _ = scaled_dot_product_attention_torch(Q, K, V, mask)

        # transpose back, ensure memory is aligned, bring back to original shape
        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        output = self.W_o(output)

        return output

In [ ]:
# processes the source sententence
class EncoderLayer(nn.Module):
    def __init__(self, d_model=64, num_heads=2, d_ff=128):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        # simple 2 layer nn applied to every token
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )

        # layer normilization to stabilize training
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask):
        # selve attention
        attn_out = self.self_attn(x, x, x, mask)
        # add and normalize
        x = self.norm1(x + attn_out)
        ff_out = self.feed_forward(x)
        x = self.norm2(x + ff_out)
        return x

In [ ]:
# generates target sentence
class DecoderLayer(nn.Module):
    def __init__(self, d_model=64, num_heads=2, d_ff=128):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, x, enc_out, src_mask, trg_mask):
        # the mask prevents us from looking at future words
        self_attn_out = self.self_attn(x, x, x, trg_mask)
        x = self.norm1(x + self_attn_out)

        # look back at the French sentence
        cross_attn_out = self.cross_attn(x, enc_out, enc_out, src_mask)
        x = self.norm2(x + cross_attn_out)

        ff_out = self.feed_forward(x)
        x = self.norm3(x + ff_out)
        return x

In [ ]:
# wraps all the parts together
class SimplifiedTransformer(nn.Module):
    def __init__(self, src_vocab_size, trg_vocab_size, d_model=64, num_heads=2, num_layers=2, d_ff=128, pad_idx=0):
        super().__init__()
        self.pad_idx = pad_idx

        # convert token IDs to dense vectors
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.trg_embedding = nn.Embedding(trg_vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        # create 2 layers instead of 6
        self.encoder_layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff) for _ in range(num_layers)])
        self.decoder_layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff) for _ in range(num_layers)])

        # final layer maps to vocab size to predict words
        self.fc_out = nn.Linear(d_model, trg_vocab_size)

    def make_src_mask(self, src):
        # mask indicating the pad tokens location
        return (src != self.pad_idx).unsqueeze(1).unsqueeze(2)

    def make_trg_mask(self, trg):
        # padding mask
        trg_pad_mask = (trg != self.pad_idx).unsqueeze(1).unsqueeze(2)
        trg_len = trg.shape[1]
        # mask to look ahead
        trg_sub_mask = torch.tril(torch.ones((trg_len, trg_len), device=trg.device)).bool()
        return trg_pad_mask & trg_sub_mask

    def forward(self, src, trg):
        # generate current batch's masks
        src_mask = self.make_src_mask(src)
        trg_mask = self.make_trg_mask(trg)

        # encode the source
        enc_src = self.pos_encoder(self.src_embedding(src))
        for layer in self.encoder_layers:
            enc_src = layer(enc_src, src_mask)

        # decode the target
        dec_trg = self.pos_encoder(self.trg_embedding(trg))
        for layer in self.decoder_layers:
            dec_trg = layer(dec_trg, enc_src, src_mask, trg_mask)

        # return the raw logits
        return self.fc_out(dec_trg)

In [ ]:
transformer_model = SimplifiedTransformer(
    src_vocab_size=INPUT_DIM,
    trg_vocab_size=OUTPUT_DIM,
    d_model=64,
    num_heads=2,
    num_layers=2,
    d_ff=128,
    pad_idx=0
).to(device)

optimizer_tf = optim.Adam(transformer_model.parameters())
criterion_tf = nn.CrossEntropyLoss(ignore_index=0)

In [ ]:
for epoch in range(epochs):
    # Using the same number of epochs as Part 3 (10)
    transformer_model.train()
    epoch_loss = 0

    for src, trg in train_loader:
        src, trg = src.to(device), trg.to(device)


        trg_input = trg[:, :-1]
        trg_expected = trg[:, 1:]

        optimizer_tf.zero_grad()
        output = transformer_model(src, trg_input)

        # reshape for CE loss
        output_dim = output.shape[-1]
        output = output.contiguous().view(-1, output_dim)
        trg_expected = trg_expected.contiguous().view(-1)

        loss = criterion_tf(output, trg_expected)
        loss.backward()
        optimizer_tf.step()

        epoch_loss += loss.item()

    print(f"Transformer Epoch: {epoch+1}, Loss: {epoch_loss/len(train_loader):.3f}")

Transformer Epoch: 1, Loss: 6.672
Transformer Epoch: 2, Loss: 5.816
Transformer Epoch: 3, Loss: 5.382
Transformer Epoch: 4, Loss: 5.078
Transformer Epoch: 5, Loss: 4.826
Transformer Epoch: 6, Loss: 4.602
Transformer Epoch: 7, Loss: 4.399
Transformer Epoch: 8, Loss: 4.212
Transformer Epoch: 9, Loss: 4.030
Transformer Epoch: 10, Loss: 3.853


In [ ]:
def greedy_decode_transformer(model, src_tensor, max_len=50):
    model.eval()
    with torch.no_grad():

        # encode source sentence
        src_mask = model.make_src_mask(src_tensor)
        enc_src = model.pos_encoder(model.src_embedding(src_tensor))
        for layer in model.encoder_layers:
            enc_src = layer(enc_src, src_mask)

        # start with the <SOS> token
        trg_indexes = [fra_vocab.stoi["<SOS>"]]

        for _ in range(max_len):
            # convert current predicted sequence into a tensor
            trg_tensor = torch.tensor(trg_indexes).unsqueeze(0).to(device)
            trg_mask = model.make_trg_mask(trg_tensor)

            # pass through the decoder
            dec_trg = model.pos_encoder(model.trg_embedding(trg_tensor))
            for layer in model.decoder_layers:
                dec_trg = layer(dec_trg, enc_src, src_mask, trg_mask)

            output = model.fc_out(dec_trg)
            # get the highest prob tocken fo rthe last word we predicted
            pred_token = output.argmax(2)[:, -1].item()
            trg_indexes.append(pred_token)

            # stop if the model thinks it's the end of the sentence
            if pred_token == fra_vocab.stoi["<EOS>"]:
                break

    return trg_indexes

In [ ]:
def evaluate_transformer_bleu(model, test_dataset, trg_vocab):
    targets = []
    outputs = []

    # process one sentence at a time
    for src_tuple, trg_tuple in test_dataset:
        # batch dimension of 1
        src_tensor = src_tuple.unsqueeze(0).to(device)
        trg_tensor = trg_tuple.unsqueeze(0).to(device)

        # get the actual words from the target to compare against
        trg_words = [trg_vocab.itos[idx.item()] for idx in trg_tensor[0] if idx.item() not in [0, 1, 2]]
        targets.append([trg_words])

        # make translation
        pred_indexes = greedy_decode_transformer(model, src_tensor)

        # convert predicted ids back to words
        pred_words = [trg_vocab.itos.get(idx, "<UNK>") for idx in pred_indexes if idx not in [0, 1, 2]]
        outputs.append(pred_words)

    return corpus_bleu(targets, outputs)

In [ ]:
tf_bleu = evaluate_transformer_bleu(transformer_model, test_dataset, fra_vocab)
print(f"Transformer BLEU Score: {tf_bleu*100:.2f}")

Transformer BLEU Score: 1.64


### Evaluation
Seq2Seq took 10 minutes to train and had a Bleu score of 0.44. The simplified transformer model took less than a minute to train and had a score of 1.64. Neither of these models performed particularly well. However, this is not a suprise. Using only 10,000 sentence pairs and training for only 10 epochs is a drop in the ocean compared to the millions of sentence pairs that are required for a model to achieve competency anywhere approaching fluency. Despite the poor performance, the transformer model performed extrodinarily when compared with the the Seq2Seq model in both score and time.

The score difference demonstrates the advantage of the transformer architecutre really well. The seq2swq model needs to compress the entire meaning a language, the English language in our case, down into a single vector beforee it can decode. Even with the implementation above of the Luong attention, the process is still very lossy. The transformer, however, uses self and cross attention, allowing the decoder to weigh the importance of each word simultaneously. This allows it preserve better context and create better translations.

The runtime difference comes down to how the models process components differently. The Seq2Seq, as indicated in its name, must process the words sequencially. This prevents parallelization that could speed up the process. The transformer on the other hand goes all in on the parallelization. It's workings mean that most of it's calculations are matrix math which GPUs are very capable of completing quickly.

Overall, with this sample size, the transformer performs better both in terms of score and speed.